# 2026-10-14 — XFeat vs 고정 분위수 thermometer + 일반 binary bin 표현 비교

학습 데이터와 분위수 양자화는 유지합니다. 기존 경계가 변하지 않았음을 검사하며, 같은 bin 개수의 두 표현은 **동일한 경계 배열**을 공유합니다. LUT, float 비트 선택, projection, k-means는 사용하지 않습니다.

| 표현 | Bin 개수 | bits/원소 | 저장 bytes/KP |
|---|---:|---:|---:|
| 공식 XFeat float cosine | — | — | 256 |
| thermometer | 8 | 7 | 56 |
| thermometer | 9 | 8 | 64 |
| thermometer | 10 | 9 | 72 |
| thermometer | 11 | 10 | 80 |
| thermometer 대조군 | 16 | 15 | 120 |
| 일반 binary 정수 ABS | 8 | 3 | 24 |
| 일반 binary 정수 ABS | 16 | 4 | 32 |

**표현만 비교:** 8-bin thermometer ↔ 3-bit binary는 동일한 7개 경계와 동일한 bin 번호를 사용합니다. 16-bin thermometer ↔ 4-bit binary는 동일한 15개 경계와 동일한 bin 번호를 사용합니다.
8-bin과 16-bin 자체를 비교하면 bin 개수에 따른 양자화 정밀도도 달라집니다. 양자화 알고리즘은 모두 동일한 차원별 고정 분위수입니다.
일반 binary는 `000`~`111`, 또는 `0000`~`1111`을 unsigned 정수로 해석하여 ABS 합을 계산합니다. 일반 binary에 XOR/Hamming을 적용하지 않습니다.

## 데이터
- COCO val2017 256장: 경계 계산 192장 / 독립 평가 원본 64장. 이미지 파일명을 먼저 분리합니다.
- Calibration의 191,581 normalized 64D descriptor로 각 차원의 경계를 한 번 계산합니다. matching loss로 경계를 학습하지 않습니다.
- 독립 COCO 64장 × 알려진 원근/회전/스케일/이동/밝기 변환 3개 = 192쌍.
- HPatches 전체 116장면 × 6장 = 696장, 기준 1번 ↔ 2~6번 = 580쌍. 조명 57장면 / 시점 59장면.
- 전체 772쌍. HPatches 데이터로 경계를 정하지 않습니다. 최대 변 640, 이미지당 최대 1,000 KP. 리사이즈에 맞게 homography도 보정합니다.

## 계산, 저장량, 검증
원본 cosine은 공식 `match(..., min_cossim=-1)`입니다. 모든 방법이 같은 descriptor/keypoint cache를 공유합니다.
Thermometer는 direct AVX2 인코딩 + SIMD4 거리 + 양방향 동시 MNN 갱신을 사용합니다.
Binary 인코딩은 같은 float 값을 같은 quantile threshold와 비교한 후 얻은 bin 번호를 3/4-bit로 packing합니다.
3-bit binary는 24 B 저장 후 매칭 호출당 한 번 64 B/KP의 작업 배열로 복원합니다. **복원 시간은 preencoded_matching과 total 시간 모두에 포함**됩니다.
4-bit binary는 32 B/KP의 packed nibble을 직접 비교하며, descriptor 전체를 64 B 배열로 풀지 않습니다.
표의 저장량은 descriptor만 포함합니다. 좌표/score/공유 경계/임시 배열/최대 작업 메모리는 별도입니다.
8-bin/16-bin에서 bin 번호, ABS 거리, MNN tie/empty/tail/threshold equality를 검사합니다. 실제 모든 772쌍에서 binary와 동일 bin thermometer의 MNN 결과 일치를 assert합니다.

## 평가와 실행
CPU 2-thread, 모든 772쌍에서 warmup 3회 + 반복 9회. 인코딩+packing+layout conversion/unpacking+MNN을 포함한 total과 preencoded matching을 표시합니다. 특징 추출은 제외합니다.
정확도는 pooled correct/mutual (<1/3/5 px). 공식 HPatches patch benchmark/MMA와 다른 자체 full-image KP 매칭 precision입니다.
Colab CPU에서 모두 실행하세요. 기존 Drive `CSE527_datasets` ZIP을 재사용합니다. 코드가 노트북에 포함되어 있고 기존 데이터 캐시도 재사용합니다.


## 실제 재실행 결과 — binary notation 추가

같은 calibration/descriptor/keypoint cache로 8개 방법을 모두 재측정했습니다. 기존 8/9/10/11-bin quantile 경계는 변경되지 않았습니다. 8-bin 및 16-bin binary/thermometer가 772쌍 모두에서 동일한 MNN 결과를 반환했습니다. 시간은 이번 재실행 값이며 이전 시간과 섞지 않습니다.

| 방법 | COCO precision | HPatches precision | HPatches encode+MNN ms | 저장 B/KP | 작업 표현 B/KP |
|---|---:|---:|---:|---:|---:|
| official_float_cosine | 70.565% | 60.589% | 3.832 | 256 | 256 |
| quantile_8bin_thermometer_AVX2 | 68.642% | 58.507% | 1.023 | 56 | 56 |
| quantile_9bin_thermometer_AVX2 | 68.883% | 58.659% | 1.163 | 64 | 64 |
| quantile_10bin_thermometer_AVX2 | 68.969% | 58.806% | 1.280 | 72 | 72 |
| quantile_11bin_thermometer_AVX2 | 69.192% | 58.913% | 1.492 | 80 | 80 |
| quantile_16bin_thermometer_AVX2 | 69.534% | 59.133% | 2.207 | 120 | 120 |
| quantile_8bin_binary_3bit_ABS | 68.642% | 58.507% | 1.987 | 24 | 64 |
| quantile_16bin_binary_4bit_ABS | 69.534% | 59.133% | 2.232 | 32 | 32 |

3-bit binary의 24B 저장은 촘촘한 packing입니다. 매칭할 때 64B 작업 배열로 한 번 복원하는 비용까지 위 시간에 포함했습니다. 4-bit binary는 32B packed 표현에서 직접 ABS를 계산합니다. 작업 표현 크기는 최대 임시 메모리 사용량과 다릅니다.

같은 bin 개수에서는 두 표현의 정확도가 같습니다. 16-bin의 정확도 변화는 표현의 변화가 아니라 bin 개수가 증가한 양자화 효과입니다. <1px / <5px precision과 correct counts도 CSV에서 확인할 수 있습니다.

```json
{
  "COCO_synthetic": [
    {
      "method": "official_float_cosine",
      "pairs": "192",
      "mutual": "109639",
      "correct_3px": "77367",
      "precision_1px": "39.94107936044656",
      "precision_3px": "70.56521858097939",
      "precision_5px": "84.9496985561707",
      "encode_ms": "0.0",
      "preencoded_matching_ms": "3.7852369998745417",
      "total_ms": "3.7710404999415914",
      "descriptor_bytes": "256",
      "working_bytes": "256",
      "representation": "float32",
      "mean_correct_3px_per_pair": "402.953125",
      "speedup_vs_float": "1.0"
    },
    {
      "method": "quantile_8bin_thermometer_AVX2",
      "pairs": "192",
      "mutual": "106832",
      "correct_3px": "73332",
      "precision_1px": "38.90875393140632",
      "precision_3px": "68.64235435075632",
      "precision_5px": "83.61726823423693",
      "encode_ms": "0.07179849990279763",
      "preencoded_matching_ms": "0.9197829999720852",
      "total_ms": "1.0058984998977394",
      "descriptor_bytes": "56",
      "working_bytes": "56",
      "representation": "thermometer",
      "mean_correct_3px_per_pair": "381.9375",
      "speedup_vs_float": "3.748927451750807"
    },
    {
      "method": "quantile_9bin_thermometer_AVX2",
      "pairs": "192",
      "mutual": "107107",
      "correct_3px": "73778",
      "precision_1px": "39.10015218426433",
      "precision_3px": "68.88251934980907",
      "precision_5px": "83.76296600595666",
      "encode_ms": "0.07888450022619509",
      "preencoded_matching_ms": "1.0539859997606982",
      "total_ms": "1.1597115001222846",
      "descriptor_bytes": "64",
      "working_bytes": "64",
      "representation": "thermometer",
      "mean_correct_3px_per_pair": "384.2604166666667",
      "speedup_vs_float": "3.251705703999622"
    },
    {
      "method": "quantile_10bin_thermometer_AVX2",
      "pairs": "192",
      "mutual": "107448",
      "correct_3px": "74106",
      "precision_1px": "39.092398183307274",
      "precision_3px": "68.9691757873576",
      "precision_5px": "83.85358498994863",
      "encode_ms": "0.08647600020594837",
      "preencoded_matching_ms": "1.1435215001256438",
      "total_ms": "1.2568330002977746",
      "descriptor_bytes": "72",
      "working_bytes": "72",
      "representation": "thermometer",
      "mean_correct_3px_per_pair": "385.96875",
      "speedup_vs_float": "3.000430844072474"
    },
    {
      "method": "quantile_11bin_thermometer_AVX2",
      "pairs": "192",
      "mutual": "107597",
      "correct_3px": "74449",
      "precision_1px": "39.27804678569105",
      "precision_3px": "69.19244960361348",
      "precision_5px": "83.98933055754343",
      "encode_ms": "0.11876450003001082",
      "preencoded_matching_ms": "1.2489564999214053",
      "total_ms": "1.4454595002462156",
      "descriptor_bytes": "80",
      "working_bytes": "80",
      "representation": "thermometer",
      "mean_correct_3px_per_pair": "387.7552083333333",
      "speedup_vs_float": "2.608887000500009"
    },
    {
      "method": "quantile_16bin_thermometer_AVX2",
      "pairs": "192",
      "mutual": "107971",
      "correct_3px": "75077",
      "precision_1px": "39.404099248872384",
      "precision_3px": "69.53441201804188",
      "precision_5px": "84.1994609663706",
      "encode_ms": "0.13358699993659684",
      "preencoded_matching_ms": "1.998146000005363",
      "total_ms": "2.1918544998698053",
      "descriptor_bytes": "120",
      "working_bytes": "120",
      "representation": "thermometer",
      "mean_correct_3px_per_pair": "391.0260416666667",
      "speedup_vs_float": "1.7204793932104474"
    },
    {
      "method": "quantile_8bin_binary_3bit_ABS",
      "pairs": "192",
      "mutual": "106832",
      "correct_3px": "73332",
      "precision_1px": "38.90875393140632",
      "precision_3px": "68.64235435075632",
      "precision_5px": "83.61726823423693",
      "encode_ms": "0.3075409999837575",
      "preencoded_matching_ms": "1.6288715000882803",
      "total_ms": "1.9362875000297208",
      "descriptor_bytes": "24",
      "working_bytes": "64",
      "representation": "ordinary_binary",
      "mean_correct_3px_per_pair": "381.9375",
      "speedup_vs_float": "1.9475622808512207"
    },
    {
      "method": "quantile_16bin_binary_4bit_ABS",
      "pairs": "192",
      "mutual": "107971",
      "correct_3px": "75077",
      "precision_1px": "39.404099248872384",
      "precision_3px": "69.53441201804188",
      "precision_5px": "84.1994609663706",
      "encode_ms": "0.32101100009640504",
      "preencoded_matching_ms": "1.8679434999739897",
      "total_ms": "2.218514499872981",
      "descriptor_bytes": "32",
      "working_bytes": "32",
      "representation": "ordinary_binary",
      "mean_correct_3px_per_pair": "391.0260416666667",
      "speedup_vs_float": "1.6998043060604286"
    }
  ],
  "HPatches": [
    {
      "method": "official_float_cosine",
      "pairs": "580",
      "mutual": "291631",
      "correct_3px": "176696",
      "precision_1px": "27.024904759782054",
      "precision_3px": "60.588894870572744",
      "precision_5px": "73.56145265763928",
      "encode_ms": "0.0",
      "preencoded_matching_ms": "3.8742075000754994",
      "total_ms": "3.8324320000810985",
      "descriptor_bytes": "256",
      "working_bytes": "256",
      "representation": "float32",
      "mean_correct_3px_per_pair": "304.64827586206894",
      "speedup_vs_float": "1.0"
    },
    {
      "method": "quantile_8bin_thermometer_AVX2",
      "pairs": "580",
      "mutual": "286965",
      "correct_3px": "167895",
      "precision_1px": "26.35060024741693",
      "precision_3px": "58.50713501646542",
      "precision_5px": "71.18707856358789",
      "encode_ms": "0.0721940000403265",
      "preencoded_matching_ms": "0.9138939999502327",
      "total_ms": "1.0231090000161203",
      "descriptor_bytes": "56",
      "working_bytes": "56",
      "representation": "thermometer",
      "mean_correct_3px_per_pair": "289.4741379310345",
      "speedup_vs_float": "3.745868719775424"
    },
    {
      "method": "quantile_9bin_thermometer_AVX2",
      "pairs": "580",
      "mutual": "287684",
      "correct_3px": "168752",
      "precision_1px": "26.44046940392931",
      "precision_3px": "58.658806190125276",
      "precision_5px": "71.40855939155462",
      "encode_ms": "0.0792849998560996",
      "preencoded_matching_ms": "1.062193500047215",
      "total_ms": "1.1631014999693434",
      "descriptor_bytes": "64",
      "working_bytes": "64",
      "representation": "thermometer",
      "mean_correct_3px_per_pair": "290.951724137931",
      "speedup_vs_float": "3.295010796720761"
    },
    {
      "method": "quantile_10bin_thermometer_AVX2",
      "pairs": "580",
      "mutual": "287925",
      "correct_3px": "169316",
      "precision_1px": "26.45133281236433",
      "precision_3px": "58.805591733958494",
      "precision_5px": "71.59989580619953",
      "encode_ms": "0.08707149982001283",
      "preencoded_matching_ms": "1.1713835001501138",
      "total_ms": "1.2795815000572475",
      "descriptor_bytes": "72",
      "working_bytes": "72",
      "representation": "thermometer",
      "mean_correct_3px_per_pair": "291.92413793103447",
      "speedup_vs_float": "2.9950667463617116"
    },
    {
      "method": "quantile_11bin_thermometer_AVX2",
      "pairs": "580",
      "mutual": "288279",
      "correct_3px": "169833",
      "precision_1px": "26.489615962314286",
      "precision_3px": "58.91271996919651",
      "precision_5px": "71.6944349050746",
      "encode_ms": "0.119265000193991",
      "preencoded_matching_ms": "1.2572134999118134",
      "total_ms": "1.4923999999609805",
      "descriptor_bytes": "80",
      "working_bytes": "80",
      "representation": "thermometer",
      "mean_correct_3px_per_pair": "292.8155172413793",
      "speedup_vs_float": "2.5679656929652235"
    },
    {
      "method": "quantile_16bin_thermometer_AVX2",
      "pairs": "580",
      "mutual": "289177",
      "correct_3px": "171000",
      "precision_1px": "26.555362286765543",
      "precision_3px": "59.13333356387265",
      "precision_5px": "71.90405875985988",
      "encode_ms": "0.13412800012702064",
      "preencoded_matching_ms": "2.037039500009996",
      "total_ms": "2.207316999829345",
      "descriptor_bytes": "120",
      "working_bytes": "120",
      "representation": "thermometer",
      "mean_correct_3px_per_pair": "294.82758620689657",
      "speedup_vs_float": "1.7362399693281012"
    },
    {
      "method": "quantile_8bin_binary_3bit_ABS",
      "pairs": "580",
      "mutual": "286965",
      "correct_3px": "167895",
      "precision_1px": "26.35060024741693",
      "precision_3px": "58.50713501646542",
      "precision_5px": "71.18707856358789",
      "encode_ms": "0.313745000084964",
      "preencoded_matching_ms": "1.6386114998567791",
      "total_ms": "1.98669700012033",
      "descriptor_bytes": "24",
      "working_bytes": "64",
      "representation": "ordinary_binary",
      "mean_correct_3px_per_pair": "289.4741379310345",
      "speedup_vs_float": "1.9290470564202677"
    },
    {
      "method": "quantile_16bin_binary_4bit_ABS",
      "pairs": "580",
      "mutual": "289177",
      "correct_3px": "171000",
      "precision_1px": "26.555362286765543",
      "precision_3px": "59.13333356387265",
      "precision_5px": "71.90405875985988",
      "encode_ms": "0.3290035001555225",
      "preencoded_matching_ms": "1.8731615000433521",
      "total_ms": "2.231518999906257",
      "descriptor_bytes": "32",
      "working_bytes": "32",
      "representation": "ordinary_binary",
      "mean_correct_3px_per_pair": "294.82758620689657",
      "speedup_vs_float": "1.7174095314635878"
    }
  ],
  "HPatches_by_type": [
    {
      "category": "illumination",
      "method": "official_float_cosine",
      "pairs": "285",
      "mutual": "166793",
      "correct_3px": "114048",
      "precision_1px": "31.513312908815116",
      "precision_3px": "68.37697025654553",
      "precision_5px": "79.59146966599317",
      "encode_ms": "0.0",
      "preencoded_matching_ms": "4.345171000295522",
      "total_ms": "4.351020000285644",
      "descriptor_bytes": "256",
      "working_bytes": "256",
      "representation": "float32",
      "mean_correct_3px_per_pair": "400.1684210526316"
    },
    {
      "category": "illumination",
      "method": "quantile_10bin_thermometer_AVX2",
      "pairs": "285",
      "mutual": "163958",
      "correct_3px": "110443",
      "precision_1px": "31.341563083228632",
      "precision_3px": "67.36054355383696",
      "precision_5px": "78.5170592468803",
      "encode_ms": "0.08749200014790404",
      "preencoded_matching_ms": "1.2043540000377106",
      "total_ms": "1.3182359998609172",
      "descriptor_bytes": "72",
      "working_bytes": "72",
      "representation": "thermometer",
      "mean_correct_3px_per_pair": "387.51929824561404"
    },
    {
      "category": "illumination",
      "method": "quantile_11bin_thermometer_AVX2",
      "pairs": "285",
      "mutual": "164222",
      "correct_3px": "110768",
      "precision_1px": "31.365468694815554",
      "precision_3px": "67.45015893120288",
      "precision_5px": "78.58325924662957",
      "encode_ms": "0.11951099986617919",
      "preencoded_matching_ms": "1.2924580000799324",
      "total_ms": "1.5348350002568623",
      "descriptor_bytes": "80",
      "working_bytes": "80",
      "representation": "thermometer",
      "mean_correct_3px_per_pair": "388.659649122807"
    },
    {
      "category": "illumination",
      "method": "quantile_16bin_binary_4bit_ABS",
      "pairs": "285",
      "mutual": "164749",
      "correct_3px": "111365",
      "precision_1px": "31.397459165154267",
      "precision_3px": "67.59676841741073",
      "precision_5px": "78.73431705200032",
      "encode_ms": "0.3308310001557402",
      "preencoded_matching_ms": "1.902290000089124",
      "total_ms": "2.2692869997626985",
      "descriptor_bytes": "32",
      "working_bytes": "32",
      "representation": "ordinary_binary",
      "mean_correct_3px_per_pair": "390.7543859649123"
    },
    {
      "category": "illumination",
      "method": "quantile_16bin_thermometer_AVX2",
      "pairs": "285",
      "mutual": "164749",
      "correct_3px": "111365",
      "precision_1px": "31.397459165154267",
      "precision_3px": "67.59676841741073",
      "precision_5px": "78.73431705200032",
      "encode_ms": "0.13456299984682119",
      "preencoded_matching_ms": "2.0674709999184415",
      "total_ms": "2.2703780000483675",
      "descriptor_bytes": "120",
      "working_bytes": "120",
      "representation": "thermometer",
      "mean_correct_3px_per_pair": "390.7543859649123"
    },
    {
      "category": "illumination",
      "method": "quantile_8bin_binary_3bit_ABS",
      "pairs": "285",
      "mutual": "163415",
      "correct_3px": "109727",
      "precision_1px": "31.304347826086957",
      "precision_3px": "67.14622280696386",
      "precision_5px": "78.23822782486307",
      "encode_ms": "0.3185119999216113",
      "preencoded_matching_ms": "1.6660430001138593",
      "total_ms": "2.0218820000081905",
      "descriptor_bytes": "24",
      "working_bytes": "64",
      "representation": "ordinary_binary",
      "mean_correct_3px_per_pair": "385.00701754385966"
    },
    {
      "category": "illumination",
      "method": "quantile_8bin_thermometer_AVX2",
      "pairs": "285",
      "mutual": "163415",
      "correct_3px": "109727",
      "precision_1px": "31.304347826086957",
      "precision_3px": "67.14622280696386",
      "precision_5px": "78.23822782486307",
      "encode_ms": "0.07245999995575403",
      "preencoded_matching_ms": "0.9405040000274312",
      "total_ms": "1.054827000189107",
      "descriptor_bytes": "56",
      "working_bytes": "56",
      "representation": "thermometer",
      "mean_correct_3px_per_pair": "385.00701754385966"
    },
    {
      "category": "illumination",
      "method": "quantile_9bin_thermometer_AVX2",
      "pairs": "285",
      "mutual": "163786",
      "correct_3px": "110069",
      "precision_1px": "31.34516991684271",
      "precision_3px": "67.20293553783596",
      "precision_5px": "78.35101901261402",
      "encode_ms": "0.07966100019984879",
      "preencoded_matching_ms": "1.0984339996866765",
      "total_ms": "1.1882700000569457",
      "descriptor_bytes": "64",
      "working_bytes": "64",
      "representation": "thermometer",
      "mean_correct_3px_per_pair": "386.20701754385965"
    },
    {
      "category": "viewpoint",
      "method": "official_float_cosine",
      "pairs": "295",
      "mutual": "124838",
      "correct_3px": "62648",
      "precision_1px": "21.028052355853184",
      "precision_3px": "50.183437735304956",
      "precision_5px": "65.50489434306861",
      "encode_ms": "0.0",
      "preencoded_matching_ms": "3.6368060000313562",
      "total_ms": "3.608268999869324",
      "descriptor_bytes": "256",
      "working_bytes": "256",
      "representation": "float32",
      "mean_correct_3px_per_pair": "212.36610169491524"
    },
    {
      "category": "viewpoint",
      "method": "quantile_10bin_thermometer_AVX2",
      "pairs": "295",
      "mutual": "123967",
      "correct_3px": "58873",
      "precision_1px": "19.983544007679463",
      "precision_3px": "47.49086450426323",
      "precision_5px": "62.45129752272782",
      "encode_ms": "0.0868410002112796",
      "preencoded_matching_ms": "1.1556099998415448",
      "total_ms": "1.2598970001818088",
      "descriptor_bytes": "72",
      "working_bytes": "72",
      "representation": "thermometer",
      "mean_correct_3px_per_pair": "199.56949152542373"
    },
    {
      "category": "viewpoint",
      "method": "quantile_11bin_thermometer_AVX2",
      "pairs": "295",
      "mutual": "124057",
      "correct_3px": "59065",
      "precision_1px": "20.035145134897668",
      "precision_3px": "47.6111787323569",
      "precision_5px": "62.57526782043738",
      "encode_ms": "0.1190199996017327",
      "preencoded_matching_ms": "1.2366329997348657",
      "total_ms": "1.466220000111207",
      "descriptor_bytes": "80",
      "working_bytes": "80",
      "representation": "thermometer",
      "mean_correct_3px_per_pair": "200.22033898305085"
    },
    {
      "category": "viewpoint",
      "method": "quantile_16bin_binary_4bit_ABS",
      "pairs": "295",
      "mutual": "124428",
      "correct_3px": "59635",
      "precision_1px": "20.144179766612016",
      "precision_3px": "47.927315395248655",
      "precision_5px": "62.86044941653004",
      "encode_ms": "0.3220180001335393",
      "preencoded_matching_ms": "1.8561179999778687",
      "total_ms": "2.2135710000839026",
      "descriptor_bytes": "32",
      "working_bytes": "32",
      "representation": "ordinary_binary",
      "mean_correct_3px_per_pair": "202.15254237288136"
    },
    {
      "category": "viewpoint",
      "method": "quantile_16bin_thermometer_AVX2",
      "pairs": "295",
      "mutual": "124428",
      "correct_3px": "59635",
      "precision_1px": "20.144179766612016",
      "precision_3px": "47.927315395248655",
      "precision_5px": "62.86044941653004",
      "encode_ms": "0.13386299997364404",
      "preencoded_matching_ms": "2.016473000367114",
      "total_ms": "2.162454999961483",
      "descriptor_bytes": "120",
      "working_bytes": "120",
      "representation": "thermometer",
      "mean_correct_3px_per_pair": "202.15254237288136"
    },
    {
      "category": "viewpoint",
      "method": "quantile_8bin_binary_3bit_ABS",
      "pairs": "295",
      "mutual": "123550",
      "correct_3px": "58168",
      "precision_1px": "19.798462161068393",
      "precision_3px": "47.080534196681505",
      "precision_5px": "61.860785107244034",
      "encode_ms": "0.30805599999439437",
      "preencoded_matching_ms": "1.6076430001703557",
      "total_ms": "1.957194000169693",
      "descriptor_bytes": "24",
      "working_bytes": "64",
      "representation": "ordinary_binary",
      "mean_correct_3px_per_pair": "197.17966101694915"
    },
    {
      "category": "viewpoint",
      "method": "quantile_8bin_thermometer_AVX2",
      "pairs": "295",
      "mutual": "123550",
      "correct_3px": "58168",
      "precision_1px": "19.798462161068393",
      "precision_3px": "47.080534196681505",
      "precision_5px": "61.860785107244034",
      "encode_ms": "0.07200899972303887",
      "preencoded_matching_ms": "0.895456000307604",
      "total_ms": "1.0055930001726665",
      "descriptor_bytes": "56",
      "working_bytes": "56",
      "representation": "thermometer",
      "mean_correct_3px_per_pair": "197.17966101694915"
    },
    {
      "category": "viewpoint",
      "method": "quantile_9bin_thermometer_AVX2",
      "pairs": "295",
      "mutual": "123898",
      "correct_3px": "58683",
      "precision_1px": "19.956738607564287",
      "precision_3px": "47.363960677331356",
      "precision_5px": "62.2310287494552",
      "encode_ms": "0.07909399982963805",
      "preencoded_matching_ms": "1.0501890001251013",
      "total_ms": "1.1449229996287613",
      "descriptor_bytes": "64",
      "working_bytes": "64",
      "representation": "thermometer",
      "mean_correct_3px_per_pair": "198.92542372881357"
    }
  ]
}
```


In [ ]:
import os
os.environ['OMP_NUM_THREADS']='2'
os.environ['MKL_NUM_THREADS']='2'
os.environ['OPENBLAS_NUM_THREADS']='2'
import sys,subprocess,importlib.util
for module,package in {'cv2':'opencv-python-headless','torch':'torch','numpy':'numpy','pandas':'pandas','tqdm':'tqdm','matplotlib':'matplotlib'}.items():
    if importlib.util.find_spec(module) is None:
        subprocess.run([sys.executable,'-m','pip','install','-q',package],check=True)
from google.colab import drive
from pathlib import Path
drive.mount('/content/drive')
BASE=Path('/content/drive/MyDrive/Colab Notebooks')
ROOT=BASE/'2026-10-14_XFeat_quantile_8_9_10_11bin_large'
ROOT.mkdir(exist_ok=True,parents=True)
COCO_ZIP=BASE/'CSE527_datasets'/'val2017.zip'
HPATCHES_ZIP=BASE/'CSE527_datasets'/'hpatches-sequences-release.zip'
assert COCO_ZIP.exists(),str(COCO_ZIP)
assert HPATCHES_ZIP.exists(),str(HPATCHES_ZIP)
COUNT=256
CALIBRATION_COUNT=192
print('저장:',ROOT)


In [ ]:
sources={'features.py': '"""Frozen XFeat extraction and dataset helpers. No LUT or bit-selection training."""\n\nfrom pathlib import Path\n\nimport subprocess,sys,zipfile\n\nimport numpy as np\n\nSEED=20261011\n\nXFEAT_COMMIT=\'e92685f57f8318b18725c5c8c0bd28c7fe188d9a\'\n\ndef project(k,H):\n    p=np.column_stack([k,np.ones(len(k))])@H.T\n    return (p[:,:2]/p[:,2:]).astype(np.float32)\n\ndef load_xfeat(root,top_k=1000):\n    import torch\n    root=Path(root);repo=root/\'xfeat_repo\'\n    if not repo.exists():\n        subprocess.run([\'git\',\'clone\',\'https://github.com/verlab/accelerated_features.git\',str(repo)],check=True)\n        subprocess.run([\'git\',\'-C\',str(repo),\'checkout\',XFEAT_COMMIT],check=True)\n    commit=subprocess.check_output([\'git\',\'-C\',str(repo),\'rev-parse\',\'HEAD\'],text=True).strip()\n    if commit!=XFEAT_COMMIT:raise RuntimeError(\'XFeat checkout differs from pinned experiment commit.\')\n    sys.path.insert(0,str(repo.resolve()))\n    from modules.xfeat import XFeat\n    torch.set_num_threads(2);torch.manual_seed(SEED)\n    model=XFeat(top_k=top_k)\n    for p in model.parameters():p.requires_grad_(False)\n    return model,commit\n\ndef extract(model,image):\n    import torch,cv2\n    t=torch.from_numpy(cv2.cvtColor(image,cv2.COLOR_BGR2RGB).copy()).permute(2,0,1)[None].float().to(model.dev)/255.\n    with torch.inference_mode():f=model.detectAndCompute(t,top_k=1000)[0]\n    d=f[\'descriptors\'].cpu().numpy().astype(np.float32)\n    d/=np.maximum(np.linalg.norm(d,axis=1,keepdims=True),1e-12)\n    return f[\'keypoints\'].cpu().numpy().astype(np.float32),np.ascontiguousarray(d)\n\ndef make_pairs(images,model,warps,seed):\n    import cv2\n    rng=np.random.default_rng(seed);pairs=[]\n    for name,img in images:\n        h,w=img.shape[:2];k1,d1=extract(model,img)\n        for r in range(warps):\n            aff=cv2.getRotationMatrix2D((w/2,h/2),float(rng.uniform(-20,20)),float(rng.uniform(.88,1.08)))\n            aff[:,2]+=rng.uniform(-.06,.06,2)*[w,h]\n            H=np.vstack([aff,[rng.uniform(-7e-5,7e-5),rng.uniform(-7e-5,7e-5),1]]).astype(np.float32)\n            im=cv2.warpPerspective(img,H,(w,h),borderMode=cv2.BORDER_CONSTANT)\n            im=np.clip(im.astype(np.float32)*rng.uniform(.82,1.12)+rng.uniform(-12,18),0,255).astype(np.uint8)\n            k2,d2=extract(model,im)\n            if min(len(d1),len(d2))<2:continue\n            pairs.append(dict(name=f\'{name}/warp{r}\',d1=d1,d2=d2,k1=k1,k2=k2,H=H))\n        print(\'extract\',name,len(d1),flush=True)\n    return pairs\n\ndef resize_decode(raw,max_side=640):\n    import cv2\n    im=cv2.imdecode(np.frombuffer(raw,np.uint8),cv2.IMREAD_COLOR)\n    if im is None:raise ValueError(\'Unreadable image\')\n    h,w=im.shape[:2];s=min(1,max_side/max(h,w));return cv2.resize(im,(max(64,int(w*s)//32*32),max(64,int(h*s)//32*32)))\n\ndef read_coco(zip_path,count=24):\n    with zipfile.ZipFile(zip_path) as z:\n        names=sorted(n for n in z.namelist() if n.lower().endswith(\'.jpg\'))\n        if len(names)<count:raise ValueError(\'Not enough COCO images\')\n        chosen=np.random.default_rng(SEED).choice(len(names),count,False)\n        return [(Path(names[i]).name,resize_decode(z.read(names[i]))) for i in chosen]\n\ndef save_pair_cache(path,pairs):\n    payload={}\n    for i,p in enumerate(pairs):\n        for k,v in p.items():payload[f\'{i}_{k}\']=np.array(v)\n    payload[\'count\']=np.array(len(pairs));np.savez_compressed(path,**payload)\n\ndef read_pair_cache(path):\n    with np.load(path,allow_pickle=False) as z:\n        return [{k:(str(z[f\'{i}_{k}\']) if k==\'name\' else z[f\'{i}_{k}\']) for k in [\'name\',\'d1\',\'d2\',\'k1\',\'k2\',\'H\']} for i in range(int(z[\'count\']))]\n', 'optimized.py': 'import numpy as np, ctypes, tempfile, subprocess\nfrom pathlib import Path\nCPU_THREADS=2\nAVX2=True\nptr=ctypes.c_void_p;integer=ctypes.c_int\ndef adr(x):return x.ctypes.data_as(ptr)\ncpp_source=\'#include <cstdint>\\n#include <vector>\\n#include <algorithm>\\n#include <immintrin.h>\\n#include <omp.h>\\nextern "C" int has_avx(){return __builtin_cpu_supports("avx512f") && __builtin_cpu_supports("avx512vpopcntdq");}\\n__attribute__((target("popcnt"))) static inline int scalar(const uint64_t*a,const uint64_t*b){int d=0;for(int k=0;k<7;k++)d+=__builtin_popcountll(a[k]^b[k]);return d;}\\n__attribute__((target("avx512f,avx512vpopcntdq"))) static int wide(const uint64_t*a,const uint64_t*b){\\n __m512i x=_mm512_maskz_loadu_epi64(0x7f,a),y=_mm512_maskz_loadu_epi64(0x7f,b);\\n return (int)_mm512_reduce_add_epi64(_mm512_popcnt_epi64(_mm512_xor_si512(x,y)));\\n}\\nextern "C" void encode_pack(const float*x,const float*e,int n,uint8_t*c,uint64_t*p){\\n for(int i=0;i<n;i++){for(int k=0;k<7;k++)p[i*7+k]=0;\\n for(int d=0;d<64;d++){int v=0;for(int k=0;k<7;k++)v+=x[i*64+d]>e[d*7+k];c[i*64+d]=v;\\n for(int k=0;k<v;k++)p[i*7+k]|=uint64_t(1)<<d;}}\\n}\\nextern "C" void matrix(const uint64_t*a,const uint64_t*b,int n,int m,int threads,uint16_t*out){\\n #pragma omp parallel for num_threads(threads) schedule(static)\\n for(int i=0;i<n;i++)for(int j=0;j<m;j++)out[i*m+j]=scalar(a+i*7,b+j*7);\\n}\\nextern "C" void fused(const uint64_t*a,const uint64_t*b,int n,int m,int mode,int threads,int*ab,int*ba){\\n if(n==0||m==0)return;\\n const bool avx=mode==1&&has_avx();\\n std::vector<int> cd(threads*m,449),ci(threads*m,n);\\n #pragma omp parallel num_threads(threads)\\n {\\n int t=omp_get_thread_num();int*ds=cd.data()+t*m;int*ids=ci.data()+t*m;\\n #pragma omp for schedule(static)\\n for(int i=0;i<n;i++){int best=449,bj=m;\\n for(int j=0;j<m;j++){int dist=0;\\n if(mode==2){for(int k=0;k<7;k++){dist+=__builtin_popcountll(a[i*7+k]^b[j*7+k]);\\n // Strict inequality preserves the smallest-index tie rule in both directions.\\n if(dist>best&&dist>ds[j])break;}}\\n else dist=avx?wide(a+i*7,b+j*7):scalar(a+i*7,b+j*7);\\n if(dist<best){best=dist;bj=j;}\\n if(dist<ds[j]||(dist==ds[j]&&i<ids[j])){ds[j]=dist;ids[j]=i;}\\n }ab[i]=bj;}\\n }\\n for(int j=0;j<m;j++){int best=449,bi=n;for(int t=0;t<threads;t++){\\n int d=cd[t*m+j],i=ci[t*m+j];if(d<best||(d==best&&i<bi)){best=d;bi=i;}}ba[j]=bi;}\\n}\\nextern "C" int has_avx2(){return __builtin_cpu_supports("avx2");}\\nextern "C" void direct_scalar(const float*x,const float*et,int n,uint64_t*p){\\n for(int i=0;i<n;i++)for(int k=0;k<7;k++){uint64_t w=0;for(int d=0;d<64;d++)w|=uint64_t(x[i*64+d]>et[k*64+d])<<d;p[i*7+k]=w;}\\n}\\nextern "C" __attribute__((target("avx2"))) void direct_avx(const float*x,const float*et,int n,uint64_t*p){\\n for(int i=0;i<n;i++)for(int k=0;k<7;k++){uint64_t w=0;\\n for(int d=0;d<64;d+=8){auto a=_mm256_loadu_ps(x+i*64+d),b=_mm256_loadu_ps(et+k*64+d);\\n w|=uint64_t(uint32_t(_mm256_movemask_ps(_mm256_cmp_ps(a,b,_CMP_GT_OQ))))<<d;}p[i*7+k]=w;}\\n}\\n__attribute__((target("avx2"))) static inline __m256i counts4(__m256i v){\\n const __m256i lut=_mm256_setr_epi8(0,1,1,2,1,2,2,3,1,2,2,3,2,3,3,4,0,1,1,2,1,2,2,3,1,2,2,3,2,3,3,4);\\n auto mask=_mm256_set1_epi8(15);\\n auto lo=_mm256_shuffle_epi8(lut,_mm256_and_si256(v,mask));\\n auto hi=_mm256_shuffle_epi8(lut,_mm256_and_si256(_mm256_srli_epi16(v,4),mask));\\n return _mm256_sad_epu8(_mm256_add_epi8(lo,hi),_mm256_setzero_si256());\\n}\\nextern "C" void transpose7(const uint64_t*p,int n,uint64_t*s){for(int k=0;k<7;k++)for(int i=0;i<n;i++)s[k*n+i]=p[i*7+k];}\\nextern "C" __attribute__((target("avx2"))) void fused4(const uint64_t*a,const uint64_t*bs,int n,int m,int threads,int*ab,int*ba){\\n if(n==0||m==0)return;\\n std::vector<int> cd(threads*m,449),ci(threads*m,n);\\n #pragma omp parallel num_threads(threads)\\n {\\n int t=omp_get_thread_num();int*ds=cd.data()+t*m;int*ids=ci.data()+t*m;\\n #pragma omp for schedule(static)\\n for(int i=0;i<n;i++){int best=449,bj=m;int j=0;\\n for(;j+4<=m;j+=4){auto sum=_mm256_setzero_si256();\\n for(int k=0;k<7;k++){auto av=_mm256_set1_epi64x(a[i*7+k]);auto bv=_mm256_loadu_si256((const __m256i*)(bs+k*m+j));sum=_mm256_add_epi64(sum,counts4(_mm256_xor_si256(av,bv)));}\\n uint64_t dist[4];_mm256_storeu_si256((__m256i*)dist,sum);\\n for(int l=0;l<4;l++){int d=dist[l],q=j+l;if(d<best){best=d;bj=q;}if(d<ds[q]||(d==ds[q]&&i<ids[q])){ds[q]=d;ids[q]=i;}}\\n }\\n for(;j<m;j++){int d=0;for(int k=0;k<7;k++)d+=__builtin_popcountll(a[i*7+k]^bs[k*m+j]);if(d<best){best=d;bj=j;}if(d<ds[j]||(d==ds[j]&&i<ids[j])){ds[j]=d;ids[j]=i;}}\\n ab[i]=bj;}\\n }\\n for(int j=0;j<m;j++){int best=449,bi=n;for(int t=0;t<threads;t++){int d=cd[t*m+j],i=ci[t*m+j];if(d<best||(d==best&&i<bi)){best=d;bi=i;}}ba[j]=bi;}\\n}\\n\'\nclass ThermometerVariant:\n def __init__(self, bins, edges):\n  self.bins=bins;self.planes=bins-1;self.bytes_per_kp=self.planes*8\n  self.name=f\'{bins} bins / {self.planes} bits: method 1+2+3 SIMD4\' if AVX2 else f\'{bins} bins / {self.planes} bits: scalar fallback\'\n  self.edges=np.ascontiguousarray(edges,dtype=np.float32)\n  self.et=np.ascontiguousarray(self.edges.T)\n  import re\n  source=re.sub(r\'\\b7\\b\',str(self.planes),cpp_source).replace(\'449\',str(64*self.planes+1))\n  # Original AVX512 distance loads at most eight planes: use exact scalar here.\n  # The benchmarked SIMD4 function is AVX2 and still processes ALL planes.\n  begin=source.index(\' __m512i x=\');end=source.index(\'\\n}\',begin)\n  source=source[:begin]+\' return scalar(a,b);\'+source[end:]\n  folder=Path(tempfile.mkdtemp(prefix=f\'thermometer_{bins}bin_\'))\n  (folder/\'kernel.cpp\').write_text(source)\n  subprocess.run([\'g++\',\'-O3\',\'-std=c++17\',\'-shared\',\'-fPIC\',\'-fopenmp\',\'-mpopcnt\',str(folder/\'kernel.cpp\'),\'-o\',str(folder/\'kernel.so\')],check=True)\n  self.lib=ctypes.CDLL(str(folder/\'kernel.so\'))\n  self.lib.has_avx2.restype=integer\n  if not self.lib.has_avx2():raise RuntimeError(\'This success method requires an AVX2 x86 CPU. Use a compatible CPU runtime.\')\n  for name in (\'direct_scalar\',\'direct_avx\'):getattr(self.lib,name).argtypes=[ptr,ptr,integer,ptr]\n  self.lib.transpose7.argtypes=[ptr,integer,ptr]\n  self.lib.fused4.argtypes=[ptr,ptr,integer,integer,integer,ptr,ptr]\n  self.lib.fused.argtypes=[ptr,ptr,integer,integer,integer,integer,ptr,ptr]\n  self.lib.matrix.argtypes=[ptr,ptr,integer,integer,integer,ptr]\n  for name in (\'direct_scalar\',\'direct_avx\',\'transpose7\',\'fused4\',\'fused\',\'matrix\'):getattr(self.lib,name).restype=None\n def codes(self,x):return np.ascontiguousarray((x[:,:,None]>self.edges[None,:,:]).sum(axis=2),dtype=np.uint8)\n def encode(self,x,simd=True):\n  x=np.ascontiguousarray(x,dtype=np.float32);out=np.empty((len(x),self.planes),np.uint64)\n  fn=self.lib.direct_avx if simd and AVX2 else self.lib.direct_scalar\n  fn(adr(x),adr(self.et),len(x),adr(out));return out\n def soa(self,p):\n  out=np.empty((self.planes,len(p)),np.uint64);self.lib.transpose7(adr(p),len(p),adr(out));return out\n def match(self,a,bs):\n  nn,mm=len(a),bs.shape[1]\n  if not nn or not mm:return np.empty(0,np.int64),np.empty(0,np.int64)\n  ab=np.empty(nn,np.int32);ba=np.empty(mm,np.int32)\n  if AVX2:self.lib.fused4(adr(a),adr(bs),nn,mm,CPU_THREADS,adr(ab),adr(ba))\n  else:\n   b=np.ascontiguousarray(bs.T);self.lib.fused(adr(a),adr(b),nn,mm,0,CPU_THREADS,adr(ab),adr(ba))\n  ii=np.arange(nn);keep=ba[ab]==ii;return ii[keep],ab[keep].astype(np.int64)\n def matrix(self,a,b):\n  out=np.empty((len(a),len(b)),np.uint16);self.lib.matrix(adr(a),adr(b),len(a),len(b),CPU_THREADS,adr(out));return out\n def full(self,p):return self.match(self.encode(p[\'x\']),self.soa(self.encode(p[\'y\'])))\n', 'binary.py': '"""Same fixed quantile bins, ordinary packed binary indices, integer ABS MNN."""\nfrom pathlib import Path\nimport ctypes,subprocess,tempfile\nimport numpy as np\n\nCPP=r\'\'\'\n#include <cstdint>\n#include <vector>\n#include <algorithm>\n#include <immintrin.h>\n#include <omp.h>\nextern "C" int has_avx2(){return __builtin_cpu_supports("avx2");}\nextern "C" __attribute__((target("avx2"))) void encode_binary(const float*x,const float*et,int n,int levels,int bits,uint8_t*out){\nint bytes=bits*8;\nfor(int i=0;i<n;i++){\n uint8_t codes[64];\n for(int d=0;d<64;d+=8){auto xx=_mm256_loadu_ps(x+i*64+d);auto c=_mm256_setzero_si256();\n for(int k=0;k<levels-1;k++){auto e=_mm256_loadu_ps(et+k*64+d);auto cmp=_mm256_cmp_ps(xx,e,_CMP_GT_OQ);c=_mm256_add_epi32(c,_mm256_srli_epi32(_mm256_castps_si256(cmp),31));}\n uint32_t values[8];_mm256_storeu_si256((__m256i*)values,c);for(int r=0;r<8;r++)codes[d+r]=values[r];}\n std::fill(out+i*bytes,out+(i+1)*bytes,0);\n for(int d=0;d<64;d++){int off=d*bits;out[i*bytes+off/8]|=unsigned(codes[d])<<(off%8);\n if(off%8+bits>8)out[i*bytes+off/8+1]|=unsigned(codes[d])>>(8-off%8);}\n}}\nextern "C" void unpack_binary(const uint8_t*x,int n,int bits,uint8_t*out){\nint bytes=bits*8;for(int i=0;i<n;i++)for(int d=0;d<64;d++){int off=d*bits;unsigned v=x[i*bytes+off/8]>>(off%8);\nif(off%8+bits>8)v|=unsigned(x[i*bytes+off/8+1])<<(8-off%8);out[i*64+d]=v&((1u<<bits)-1);}}\nstatic inline unsigned distance_bytes(const uint8_t*a,const uint8_t*b){\nauto sum=_mm_setzero_si128();for(int d=0;d<64;d+=16){auto x=_mm_loadu_si128((const __m128i*)(a+d)),y=_mm_loadu_si128((const __m128i*)(b+d));\nsum=_mm_add_epi64(sum,_mm_sad_epu8(_mm_sub_epi8(_mm_max_epu8(x,y),_mm_min_epu8(x,y)),_mm_setzero_si128()));}\nuint64_t s[2];_mm_storeu_si128((__m128i*)s,sum);return s[0]+s[1];}\nstatic inline unsigned distance_nibbles(const uint8_t*a,const uint8_t*b){\nauto sum=_mm_setzero_si128(),mask=_mm_set1_epi8(15),zero=_mm_setzero_si128();\nfor(int d=0;d<32;d+=16){auto x=_mm_loadu_si128((const __m128i*)(a+d)),y=_mm_loadu_si128((const __m128i*)(b+d));\nauto xl=_mm_and_si128(x,mask),yl=_mm_and_si128(y,mask),xh=_mm_and_si128(_mm_srli_epi16(x,4),mask),yh=_mm_and_si128(_mm_srli_epi16(y,4),mask);\nsum=_mm_add_epi64(sum,_mm_sad_epu8(_mm_sub_epi8(_mm_max_epu8(xl,yl),_mm_min_epu8(xl,yl)),zero));\nsum=_mm_add_epi64(sum,_mm_sad_epu8(_mm_sub_epi8(_mm_max_epu8(xh,yh),_mm_min_epu8(xh,yh)),zero));}\nuint64_t s[2];_mm_storeu_si128((__m128i*)s,sum);return s[0]+s[1];}\nextern "C" void match_binary(const uint8_t*ap,const uint8_t*bp,int n,int m,int bits,int threads,int*ab,int*ba){\nif(!n||!m)return;\nstd::vector<uint8_t>au,bu;const uint8_t*a=ap,*b=bp;int width=32;\nif(bits==3){au.resize(n*64);bu.resize(m*64);unpack_binary(ap,n,bits,au.data());unpack_binary(bp,m,bits,bu.data());a=au.data();b=bu.data();width=64;}\nstd::vector<int> cd(threads*m,961),ci(threads*m,n);\n#pragma omp parallel num_threads(threads)\n{\nint t=omp_get_thread_num();int*ds=cd.data()+t*m;int*ids=ci.data()+t*m;\n#pragma omp for schedule(static)\nfor(int i=0;i<n;i++){int best=961,bj=0;\nfor(int j=0;j<m;j++){int v=bits==4?distance_nibbles(a+i*width,b+j*width):distance_bytes(a+i*width,b+j*width);\nif(v<best){best=v;bj=j;}if(v<ds[j]||(v==ds[j]&&i<ids[j])){ds[j]=v;ids[j]=i;}}\nab[i]=bj;}}\nfor(int j=0;j<m;j++){int best=961,bi=n;for(int t=0;t<threads;t++){int d=cd[t*m+j],i=ci[t*m+j];if(d<best||(d==best&&i<bi)){best=d;bi=i;}}ba[j]=bi;}\n}\nextern "C" void distance_binary(const uint8_t*ap,const uint8_t*bp,int n,int m,int bits,uint16_t*out){\nstd::vector<uint8_t>au(n*64),bu(m*64);unpack_binary(ap,n,bits,au.data());unpack_binary(bp,m,bits,bu.data());\nfor(int i=0;i<n;i++)for(int j=0;j<m;j++)out[i*m+j]=distance_bytes(au.data()+i*64,bu.data()+j*64);\n}\n\'\'\'\nclass BinaryVariant:\n kind=\'binary\'\n def __init__(self,bins,edges):\n  assert bins in (8,16)\n  self.bins=bins;self.bits=3 if bins==8 else 4;self.edges=np.ascontiguousarray(edges,np.float32);self.et=np.ascontiguousarray(self.edges.T)\n  self.bytes_per_kp=self.bits*8;self.working_bytes_per_kp=64 if self.bits==3 else 32\n  folder=Path(tempfile.mkdtemp(prefix=\'quantile_binary_\'));(folder/\'kernel.cpp\').write_text(CPP)\n  subprocess.run([\'g++\',\'-O3\',\'-std=c++17\',\'-shared\',\'-fPIC\',\'-fopenmp\',str(folder/\'kernel.cpp\'),\'-o\',str(folder/\'kernel.so\')],check=True)\n  self.lib=ctypes.CDLL(str(folder/\'kernel.so\'));P=ctypes.c_void_p;I=ctypes.c_int\n  self.lib.has_avx2.restype=I\n  if not self.lib.has_avx2():raise RuntimeError(\'AVX2 CPU required for encoding\')\n  for name,args in {\'encode_binary\':[P,P,I,I,I,P],\'unpack_binary\':[P,I,I,P],\'match_binary\':[P,P,I,I,I,I,P,P],\'distance_binary\':[P,P,I,I,I,P]}.items():\n   getattr(self.lib,name).argtypes=args;getattr(self.lib,name).restype=None\n @staticmethod\n def ptr(a):return a.ctypes.data_as(ctypes.c_void_p)\n def encode(self,x):\n  x=np.ascontiguousarray(x,np.float32);out=np.empty((len(x),self.bytes_per_kp),np.uint8)\n  self.lib.encode_binary(self.ptr(x),self.ptr(self.et),len(x),self.bins,self.bits,self.ptr(out));return out\n def decode(self,p):\n  p=np.ascontiguousarray(p,np.uint8);out=np.empty((len(p),64),np.uint8);self.lib.unpack_binary(self.ptr(p),len(p),self.bits,self.ptr(out));return out\n def match(self,a,b):\n  a=np.ascontiguousarray(a,np.uint8);b=np.ascontiguousarray(b,np.uint8)\n  if not len(a) or not len(b):return np.empty(0,int),np.empty(0,int)\n  ab=np.empty(len(a),np.int32);ba=np.empty(len(b),np.int32)\n  self.lib.match_binary(self.ptr(a),self.ptr(b),len(a),len(b),self.bits,2,self.ptr(ab),self.ptr(ba))\n  i=np.flatnonzero(ba[ab]==np.arange(len(a)));return i,ab[i]\n def distance(self,a,b):\n  out=np.empty((len(a),len(b)),np.uint16);self.lib.distance_binary(self.ptr(a),self.ptr(b),len(a),len(b),self.bits,self.ptr(out));return out\n\ndef verify_binary(binary,thermo):\n assert np.array_equal(binary.edges,thermo.edges)\n rng=np.random.default_rng(20261014+binary.bins)\n for n,m in [(0,3),(3,0),(1,1),(13,19),(7,9)]:\n  x=rng.normal(size=(n,64)).astype(np.float32);y=rng.normal(size=(m,64)).astype(np.float32)\n  for tied in [False,True]:\n   if tied:x[:]=0;y[:]=0\n   a,b=binary.encode(x),binary.encode(y)\n   assert np.array_equal(binary.decode(a),thermo.codes(x));assert np.array_equal(binary.decode(b),thermo.codes(y))\n   expected=np.abs(thermo.codes(x)[:,None,:].astype(np.int16)-thermo.codes(y)[None,:,:].astype(np.int16)).sum(-1)\n   assert np.array_equal(binary.distance(a,b),expected)\n   for i,j in zip(binary.match(a,b),thermo.match(thermo.encode(x),thermo.soa(thermo.encode(y)))):assert np.array_equal(i,j)\n for x in [thermo.et,np.nextafter(thermo.et,-np.inf),np.nextafter(thermo.et,np.inf)]:\n  assert np.array_equal(binary.decode(binary.encode(x)),thermo.codes(x))\n print(\'binary representation verified\',binary.bins,flush=True)\n', 'hpatches_source.py': "from pathlib import Path\nimport json,time,zipfile,io,urllib.request\nimport numpy as np\nimport cv2\nimport features as e\n\n\nURL='https://huggingface.co/datasets/vbalnt/hpatches/resolve/main/hpatches-sequences-release.zip'\ndef hpatches_all(zip_path,model,cache_dir):\n cache_dir=Path(cache_dir);cache_dir.mkdir(parents=True,exist_ok=True)\n if (cache_dir/'hpatches_pairs.npz').exists():return e.read_pair_cache(cache_dir/'hpatches_pairs.npz')\n pairs=[];manifest=[];start=time.perf_counter()\n def decode(raw):\n  orig=cv2.imdecode(np.frombuffer(raw,np.uint8),cv2.IMREAD_COLOR)\n  if orig is None:raise ValueError('HPatches image decode failed')\n  img=e.resize_decode(raw,640)\n  scale=np.diag([img.shape[1]/orig.shape[1],img.shape[0]/orig.shape[0],1.]).astype(np.float64)\n  return img,scale\n with zipfile.ZipFile(zip_path) as z:\n  names=set(z.namelist());folders=sorted(n.rsplit('/',1)[0] for n in names if n.endswith('/1.ppm'))\n  for folder in folders:\n   scene=folder.split('/')[-1]\n   if not scene.startswith(('i_','v_')):continue\n   im,S1=decode(z.read(folder+'/1.ppm'));k1,d1=e.extract(model,im)\n   for idx in range(2,7):\n    raw=folder+f'/{idx}.ppm';hname=folder+f'/H_1_{idx}'\n    if raw not in names or hname not in names:raise ValueError(f'Missing image/homography: {scene}/{idx}')\n    im2,S2=decode(z.read(raw));k2,d2=e.extract(model,im2)\n    H0=np.loadtxt(io.BytesIO(z.read(hname)));H=S2@H0@np.linalg.inv(S1)\n    pairs.append(dict(name=f'{scene}/1-{idx}',d1=d1,d2=d2,k1=k1,k2=k2,H=H))\n   manifest.append(scene);print('extract',len(manifest),scene,flush=True)\n e.save_pair_cache(cache_dir/'hpatches_pairs.npz',pairs)\n (cache_dir/'hpatches_manifest.json').write_text(json.dumps(dict(scenes=manifest,n_scenes=len(manifest),n_pairs=len(pairs),illumination_scenes=sum(x.startswith('i_') for x in manifest),viewpoint_scenes=sum(x.startswith('v_') for x in manifest),max_side=640,top_k=1000,resize_homography='S2 @ H @ inv(S1)',source=URL,extraction_seconds=time.perf_counter()-start,protocol='all archive sequences, 1 versus 2..6; no excluded sequences; precision <3px, not official MMA benchmark',no_hpatches_training=True),indent=2))\n return pairs\n", 'evaluate.py': "from pathlib import Path\nimport json,time,hashlib,platform,sys,subprocess\nimport numpy as np\nimport pandas as pd\nimport torch\nimport features as f\nfrom optimized import ThermometerVariant\nfrom binary import BinaryVariant,verify_binary\n\nBINS=(8,9,10,11,16)\nSEED=20261014\nREPEATS=9\nWARMUP=3\n\ndef median_ms(fn):\n for _ in range(WARMUP):fn()\n values=[]\n for _ in range(REPEATS):\n  s=time.perf_counter();fn();values.append(1000*(time.perf_counter()-s))\n return float(np.median(values))\n\ndef verify(v):\n rng=np.random.default_rng(SEED+v.bins)\n for n,m in [(0,3),(3,0),(1,1),(13,19),(8,3)]:\n  a=rng.normal(size=(n,64)).astype(np.float32);b=rng.normal(size=(m,64)).astype(np.float32)\n  for tied in [False,True]:\n   if tied:a[:]=0;b[:]=0\n   pa,pb=v.encode(a),v.encode(b)\n   assert np.array_equal(pa,v.encode(a,False));assert np.array_equal(pb,v.encode(b,False))\n   ca,cb=v.codes(a),v.codes(b)\n   d=np.abs(ca[:,None,:].astype(np.int16)-cb[None,:,:].astype(np.int16)).sum(-1)\n   assert np.array_equal(d,v.matrix(pa,pb))\n   ii,jj=v.match(pa,v.soa(pb))\n   if n and m:\n    row=d.argmin(1);col=d.argmin(0);ids=np.flatnonzero(col[row]==np.arange(n))\n    assert np.array_equal(ii,ids);assert np.array_equal(jj,row[ids])\n   else:assert len(ii)==len(jj)==0\n for a in [v.et,np.nextafter(v.et,-np.inf),np.nextafter(v.et,np.inf)]:\n  assert np.array_equal(v.encode(a),v.encode(a,False))\n print('kernel verified',v.bins,flush=True)\n\ndef prepare_coco(zip_path,model,root,count=256,calibration_count=192):\n root=Path(root);root.mkdir(exist_ok=True,parents=True)\n path=root/'manifest.json'\n if path.exists():\n  man=json.loads(path.read_text())\n  if man['count']!=count or man['calibration_count']!=calibration_count or man['seed']!=SEED:raise ValueError('Use a new root for changed data settings')\n if (root/'calibration.npy').exists() and (root/'test_pairs.npz').exists():\n  return np.load(root/'calibration.npy'),f.read_pair_cache(root/'test_pairs.npz'),json.loads(path.read_text())\n images=sorted(f.read_coco(zip_path,count),key=lambda x:x[0])\n ids=np.random.default_rng(SEED).permutation(count);train=ids[:calibration_count];test=ids[calibration_count:]\n man=dict(source='COCO val2017',count=count,calibration_count=calibration_count,test_source_count=len(test),seed=SEED,image_selection_seed=f.SEED,calibration_files=[images[i][0] for i in train],test_files=[images[i][0] for i in test],top_k=1000,max_side=640,test_warps_per_image=3,calibration_augmentation=False)\n assert not set(man['calibration_files'])&set(man['test_files'])\n path.write_text(json.dumps(man,indent=2));desc=[];start=time.perf_counter()\n for num,i in enumerate(train):\n  _,d=f.extract(model,images[i][1]);desc.append(d)\n  if num%24==0:print('COCO calibration extraction',num+1,len(train),flush=True)\n calibration=np.ascontiguousarray(np.concatenate(desc),np.float32)\n np.save(root/'calibration.npy',calibration)\n pairs=f.make_pairs([images[i] for i in test],model,3,SEED+1)\n f.save_pair_cache(root/'test_pairs.npz',pairs)\n man['test_pairs']=len(pairs);man['calibration_descriptors']=len(calibration);man['extraction_seconds']=time.perf_counter()-start\n man['calibration_sha256']=hashlib.sha256(calibration.tobytes()).hexdigest()\n path.write_text(json.dumps(man,indent=2));return calibration,pairs,man\n\ndef fit_variants(calibration,root):\n root=Path(root);times={};edges={}\n for bins in BINS:\n  s=time.perf_counter()\n  edges[bins]=np.ascontiguousarray(np.quantile(calibration,np.arange(1,bins)/bins,axis=0).T,np.float32)\n  times[bins]=time.perf_counter()-s\n old_path=root/'fixed_quantile_edges.npz'\n if old_path.exists():\n  with np.load(old_path) as previous:\n   for k in (8,9,10,11):\n    if f'bins_{k}' in previous:assert np.array_equal(previous[f'bins_{k}'],edges[k]),'Existing quantization changed'\n np.savez(old_path,**{f'bins_{k}':v for k,v in edges.items()})\n variants={k:ThermometerVariant(k,edges[k]) for k in BINS}\n for v in variants.values():verify(v)\n for k in (8,16):\n  binary=BinaryVariant(k,edges[k]);verify_binary(binary,variants[k]);variants[f'binary_{k}']=binary\n return variants,times\n\ndef evaluate(pairs,variants,model,out,label):\n out=Path(out);out.mkdir(exist_ok=True,parents=True);rows=[]\n for ix,p in enumerate(pairs):\n  a=np.ascontiguousarray(p['d1'],np.float32);b=np.ascontiguousarray(p['d2'],np.float32)\n  methods=[('official_float_cosine',None)]+[(f'quantile_{k}bin_thermometer_AVX2',variants[k]) for k in BINS]+[(f'quantile_{k}bin_binary_{3 if k==8 else 4}bit_ABS',variants[f'binary_{k}']) for k in (8,16)]\n  matches={}\n  # Rotate benchmark order to avoid always placing the same method first.\n  shift=ix%len(methods);methods=methods[shift:]+methods[:shift]\n  for name,v in methods:\n   if v is None:\n    ta,tb=torch.from_numpy(a),torch.from_numpy(b)\n    def full():\n     with torch.inference_mode():return model.match(ta,tb,min_cossim=-1)\n    i,j=full();i=np.asarray(i);j=np.asarray(j);enc_ms=0.;match_ms=median_ms(full);total_ms=median_ms(full);bins=0;byte=256;working=256;representation='float32'\n   elif getattr(v,'kind',None)=='binary':\n    pa,pb=v.encode(a),v.encode(b);i,j=v.match(pa,pb);bins=v.bins;byte=v.bytes_per_kp;working=v.working_bytes_per_kp;representation='ordinary_binary'\n    def full():return v.match(v.encode(a),v.encode(b))\n    enc_ms=median_ms(lambda:(v.encode(a),v.encode(b)))\n    match_ms=median_ms(lambda:v.match(pa,pb));total_ms=median_ms(full)\n   else:\n    pa,pb=v.encode(a),v.encode(b);bs=v.soa(pb)\n    i,j=v.match(pa,bs);bins=v.bins;byte=8*(bins-1);working=byte;representation='thermometer'\n    def full():return v.match(v.encode(a),v.soa(v.encode(b)))\n    enc_ms=median_ms(lambda:(v.encode(a),v.soa(v.encode(b))))\n    match_ms=median_ms(lambda:v.match(pa,bs));total_ms=median_ms(full)\n    # For every pair check scalar bit-distance reference MNN against SIMD4.\n    n,m=len(pa),len(pb)\n    ab=np.empty(n,np.int32);ba=np.empty(m,np.int32)\n    if n and m:\n     import ctypes\n     P=ctypes.c_void_p\n     v.lib.fused(pa.ctypes.data_as(P),pb.ctypes.data_as(P),n,m,0,2,ab.ctypes.data_as(P),ba.ctypes.data_as(P))\n     ids=np.flatnonzero(ba[ab]==np.arange(n));assert np.array_equal(i,ids);assert np.array_equal(j,ab[ids])\n   matches[name]=(i,j)\n   err=np.linalg.norm(f.project(p['k1'][i],p['H'])-p['k2'][j],axis=1)\n   category='illumination' if p['name'].split('/')[0].startswith('i_') else 'viewpoint' if p['name'].split('/')[0].startswith('v_') else 'synthetic'\n   rows.append(dict(dataset=label,category=category,pair=p['name'],method=name,bins=bins,n=len(a),m=len(b),mutual=len(i),correct_1px=int((err<1).sum()),correct_3px=int((err<3).sum()),correct_5px=int((err<5).sum()),encode_ms=enc_ms,preencoded_matching_ms=match_ms,total_ms=total_ms,descriptor_bytes=byte,working_bytes=working,representation=representation))\n  for k in (8,16):\n   thermo=matches[f'quantile_{k}bin_thermometer_AVX2'];binary=matches[f'quantile_{k}bin_binary_{3 if k==8 else 4}bit_ABS']\n   for x,y in zip(thermo,binary):assert np.array_equal(x,y),'Encoding representation changed MNN'\n  if ix%20==0:print('evaluate',label,ix+1,len(pairs),flush=True)\n df=pd.DataFrame(rows);df.to_csv(out/'per_pair.csv',index=False)\n def summarize(g):\n  result=dict(pairs=len(g),mutual=int(g.mutual.sum()),correct_3px=int(g.correct_3px.sum()))\n  for px in [1,3,5]:result[f'precision_{px}px']=100*g[f'correct_{px}px'].sum()/g.mutual.sum() if g.mutual.sum() else np.nan\n  for k in ['encode_ms','preencoded_matching_ms','total_ms']:result[k]=float(g[k].median())\n  result['descriptor_bytes']=int(g.descriptor_bytes.iloc[0]);result['working_bytes']=int(g.working_bytes.iloc[0]);result['representation']=g.representation.iloc[0];result['mean_correct_3px_per_pair']=float(g.correct_3px.mean());return result\n summary=pd.DataFrame([dict(method=name,**summarize(g)) for name,g in df.groupby('method',sort=True)])\n order=['official_float_cosine']+[f'quantile_{k}bin_thermometer_AVX2' for k in BINS]+[f'quantile_{k}bin_binary_{3 if k==8 else 4}bit_ABS' for k in (8,16)]\n summary=summary.set_index('method').loc[order].reset_index()\n summary['speedup_vs_float']=summary.iloc[0].total_ms/summary.total_ms\n summary.to_csv(out/'summary.csv',index=False)\n by_type=pd.DataFrame([dict(category=c,method=n,**summarize(g)) for (c,n),g in df.groupby(['category','method'])])\n by_type.to_csv(out/'by_type.csv',index=False)\n print(label,summary.to_string(index=False),flush=True);return summary,df\n\ndef run(root,coco_zip,hpatches_zip=None,hpatches_cache=None,count=256,calibration_count=192):\n root=Path(root);root.mkdir(exist_ok=True,parents=True)\n # Reuse a verified local XFeat checkout if supplied by the caller.\n model,commit=f.load_xfeat(root)\n cal,coco,man=prepare_coco(coco_zip,model,root/'coco',count,calibration_count)\n variants,fit_seconds=fit_variants(cal,root)\n # Do not use HPatches in boundary fitting or parameter selection.\n if hpatches_cache and Path(hpatches_cache).exists():\n  hp=f.read_pair_cache(hpatches_cache)\n else:\n  from hpatches_source import hpatches_all\n  hp=hpatches_all(hpatches_zip,model,root/'hpatches_cache')\n assert len(hp)==580,'Expected all 116 HPatches sequences, reference versus 2..6'\n summaries={};all_records=[]\n for label,pairs in [('COCO_synthetic',coco),('HPatches',hp)]:\n  summary,records=evaluate(pairs,variants,model,root/label,label);summaries[label]=summary;all_records.append(records)\n env=dict(date='2026-10-14',seed=SEED,xfeat_commit=commit,torch_version=torch.__version__,numpy=np.__version__,platform=platform.platform(),torch_threads=2,cpp_threads=2,repeats=REPEATS,warmup=WARMUP,bins=list(BINS),fit_seconds=fit_seconds,calibration=man,scope='CPU descriptor encoding + transpose + exact MNN; XFeat extraction excluded; timing every evaluated pair',metric='pooled precision (<1/3/5 px homography error), not official HPatches MMA',hpatches_count=580,lut=False,projection=False,quantile_fitted_once=True,binary_comparison='Only bin notation changes; same per-dimension fixed quantiles and integer ABS; no XOR Hamming on ordinary binary',binary3_unpack='Once per matching call for both descriptor sets; included in preencoded and total time',binary4_direct='Packed nibble ABS without unpacked descriptor arrays',equivalence_verified_pairs=772)\n (root/'experiment_manifest.json').write_text(json.dumps(env,indent=2))\n return summaries\n"}
for name,content in sources.items():(ROOT/name).write_text(content)
sys.path.insert(0,str(ROOT))
for name in ['features','optimized','binary','hpatches_source','evaluate']:sys.modules.pop(name,None)
import evaluate


In [ ]:
summaries=evaluate.run(ROOT,COCO_ZIP,hpatches_zip=HPATCHES_ZIP,count=COUNT,calibration_count=CALIBRATION_COUNT)
from IPython.display import display
import pandas as pd
pd.set_option('display.max_columns',None)
for dataset,summary in summaries.items():
    print(dataset)
    display(summary.round(4))
    display(pd.read_csv(ROOT/dataset/'by_type.csv').round(4))
print('고정 경계:',ROOT/'fixed_quantile_edges.npz')
print('정확도/시간 쌍별 CSV:',ROOT/'HPatches'/'per_pair.csv')


In [ ]:
import matplotlib.pyplot as plt
fig,axes=plt.subplots(1,2,figsize=(18,5))
labels=['float','8 thermo','9 thermo','10 thermo','11 thermo','16 thermo','8 binary','16 binary']
for ax,(dataset,summary) in zip(axes,summaries.items()):
    ax.bar(labels,summary['precision_3px']);ax.set_title(dataset+' precision (<3 px)');ax.set_ylabel('%');ax.tick_params(axis='x',rotation=45)
fig.tight_layout();fig.savefig(ROOT/'precision_comparison.png',dpi=180);plt.show()
